In [1]:

import pandas as pd
import warnings
warnings.simplefilter('ignore')

In [2]:
df = pd.read_csv(r"C:\Users\divya\My Projects\Fraudulen\Imbalance clean_data.csv")
df

,Transaction_Amount,Time_of_Transaction,Previous_Fraudulent_Transactions,Account_Age,Number_of_Transactions_Last_24H,Fraudulent,Transaction_Type_Bank Transfer,Transaction_Type_Bill Payment,Transaction_Type_Online Purchase,Transaction_Type_POS Payment,...,Location_Houston,Location_Los Angeles,Location_Miami,Location_New York,Location_San Francisco,Location_Seattle,Payment_Method_Debit Card,Payment_Method_Invalid Method,Payment_Method_Net Banking,Payment_Method_UPI
0,16.478244,16,0,119,13,0,0,0,0,0,...,0,0,0,0,1,0,1,0,0,0
1,17.308790,13,4,79,3,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
2,19.383884,12,3,115,9,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,7.733225,15,4,3,4,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,1
4,17.116802,19,2,57,7,0,0,0,0,1,...,0,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50995,20.733860,15,0,7,8,0,0,1,0,0,...,0,0,0,1,0,0,1,0,0,0
50996,20.357504,3,1,75,11,1,0,0,1,0,...,0,0,1,0,0,0,0,0,1,0
50997,18.970581,18,3,73,5,0,0,0,0,1,...,0,0,0,0,1,0,0,0,0,0
50998,23.119651,19,2,108,14,0,0,0,0,1,...,0,0,0,1,0,0,0,0,1,0


In [3]:
X = df.drop(columns=['Fraudulent'])
y = df['Fraudulent']

In [4]:
from sklearn.model_selection import train_test_split
X_train, X_test , y_train , y_test = train_test_split(X, y, test_size=0.2, random_state=True)

In [5]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)


In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test= scaler.transform(X_test)

In [7]:
X_train

array([[ 0.67926449, -0.50586926,  1.6305834 , ..., -0.12913411,
        -0.41762316, -0.49525628],
       [-1.06044865, -0.356898  , -0.61511941, ...,  7.74388744,
        -0.41762316, -0.49525628],
       [ 0.83212649, -0.50586926,  1.6305834 , ..., -0.12913411,
         2.39450322, -0.49525628],
       ...,
       [ 0.4403301 , -0.05895547, -1.36368701, ..., -0.12913411,
        -0.41762316, -0.49525628],
       [ 0.30466976,  0.98384335, -1.36368701, ..., -0.12913411,
        -0.41762316, -0.49525628],
       [-0.63840159,  0.98384335, -0.61511941, ..., -0.12913411,
        -0.41762316, -0.49525628]])

**Manchine Learning Modelling & Evaluation**


 **1.Logistic Regression**

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

log_model = LogisticRegression()
log_model.fit(X_train,y_train)

ypred_train = log_model.predict(X_train)
ypred_test = log_model.predict(X_test)

print("Train Accuracy:",accuracy_score(y_train,ypred_train))
print("Cross validation Score:",cross_val_score(log_model,X_train,y_train,cv=5,scoring='accuracy').mean())
print("Test Accuracy:",accuracy_score(y_test,ypred_test))

Train Accuracy: 0.886396433541205
Cross validation Score: 0.8857529943617795
Test Accuracy: 0.8537254901960785


**2. Decision Tree**

In [11]:
from sklearn.tree import DecisionTreeClassifier
dt_default = DecisionTreeClassifier(random_state=True)
dt_default.fit(X_train,y_train)

#Prediction
pred_train = dt_default.predict(X_train)
base_pred  = dt_default.predict(X_test)

#Evaluation
from sklearn.metrics import accuracy_score
print("Train accuracy:",accuracy_score(pred_train,y_train))
print("Test accuracy:",accuracy_score(base_pred,y_test))

from sklearn.model_selection import cross_val_score
print("Cross Validation Score:",cross_val_score(dt_default,X,y,cv=5).mean())

Train accuracy: 1.0
Test accuracy: 0.8580392156862745
Cross Validation Score: 0.8960588235294118


**HyperParameter Tuning**

In [13]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

# Define the Decision Tree model
dt = DecisionTreeClassifier(random_state=42)

# Define hyperparameter grid
param_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [1, 2, 3, 4, 5]
}

# Perform Grid Search
grid = GridSearchCV(dt, param_grid, scoring="accuracy", cv=5)
grid.fit(X_train, y_train)

# Get the best parameters
best_params = grid.best_params_
print("Best Parameters:", best_params)

# Get the best model and evaluate
best_dt = grid.best_estimator_
train = best_dt.score(X_train, y_train)
test = best_dt.score(X_test, y_test)

print("Optimized Train Accuracy:", train)  #optimized means the model has been improved
print("Optimized Test Accuracy:", test)


Best Parameters: {'criterion': 'gini', 'max_depth': 5}
Optimized Train Accuracy: 0.7541488429624285
Optimized Test Accuracy: 0.7328431372549019


**Importance of each feature given by this model**

In [15]:
grid.best_estimator_.feature_importances_

array([0.0003952 , 0.00108127, 0.00670279, 0.00158055, 0.00106621,
       0.02111451, 0.00705164, 0.        , 0.        , 0.24870237,
       0.18541687, 0.        , 0.        , 0.        , 0.        ,
       0.00050349, 0.        , 0.        , 0.00065185, 0.16375784,
       0.        , 0.1561655 , 0.20580991])

In [16]:
feats = pd.DataFrame(data=grid.best_estimator_.feature_importances_,
                     index=X.columns,
                     columns=['Feature Importance'])
feats_imp = feats[feats['Feature Importance']>0]

important_features_list = feats_imp.index.to_list()
important_features_list

['Transaction_Amount',
 'Time_of_Transaction',
 'Previous_Fraudulent_Transactions',
 'Account_Age',
 'Number_of_Transactions_Last_24H',
 'Transaction_Type_Bank Transfer',
 'Transaction_Type_Bill Payment',
 'Device_Used_Mobile',
 'Device_Used_Tablet',
 'Location_Miami',
 'Location_Seattle',
 'Payment_Method_Debit Card',
 'Payment_Method_Net Banking',
 'Payment_Method_UPI']

**3.Random Forest**

In [18]:
#Random Forest Classifier with default parameters
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(random_state=True)
model.fit(X_train,y_train)

#Prediction 
ypred_train = model.predict(X_train)
ypred_test = model.predict(X_test)

#Evalulation
from sklearn.metrics import accuracy_score
print("Train accuracy:",accuracy_score(ypred_train,y_train))
print("Test accuracy:",accuracy_score(ypred_test,y_test))

from sklearn.model_selection import cross_val_score
print("Cross Validation Score:",cross_val_score(model,X,y,cv=5).mean())

Train accuracy: 0.9999871153945267
Test accuracy: 0.9224509803921569
Cross Validation Score: 0.9521176470588235


**HyperParameter Tuning**

In [20]:
from sklearn.model_selection import GridSearchCV

#model
estimator = RandomForestClassifier(random_state=True)

#Parameters (which you want to tune and identify the best)
param_grid = {'n_estimators':list(range(1,30))}

grid = GridSearchCV(estimator,param_grid, scoring='accuracy',cv=5)
grid.fit(X_train,y_train)
grid.best_params_

{'n_estimators': 28}

**Importance of each feature given by this model**

In [22]:
grid.best_estimator_.feature_importances_

array([0.08592895, 0.07001691, 0.03944682, 0.09321881, 0.06143909,
       0.05034995, 0.04690549, 0.04924069, 0.04536909, 0.03789237,
       0.04728639, 0.00711429, 0.02559338, 0.02797263, 0.02722589,
       0.02590588, 0.0259318 , 0.02583938, 0.02679446, 0.059684  ,
       0.01085278, 0.05455129, 0.05543966])

In [23]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


# Input with important features
X_imp = X[important_features_list]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_imp, y, test_size=0.2, random_state=True)

# Model with best hyperparameters
final_rf_model = RandomForestClassifier(n_estimators=28, random_state=True)
final_rf_model.fit(X_train, y_train)

# Predictions
ypred_train = final_rf_model.predict(X_train)
ypred_test = final_rf_model.predict(X_test)

# Evaluation
print("Train Accuracy:", accuracy_score(y_train, ypred_train))
print("Test Accuracy:", accuracy_score(y_test, ypred_test))
print("Cross Validation Score:", cross_val_score(final_rf_model, X_imp, y, cv=5).mean())




Train Accuracy: 0.9964460784313726
Test Accuracy: 0.9507843137254902
Cross Validation Score: 0.9520392156862746


**Adaboost**

**Applying Hyperparameter Tuning for Identifying best parameters for adaboost**



In [25]:
from sklearn.model_selection import GridSearchCV

#model/estimator
from sklearn.ensemble import AdaBoostClassifier
estimator_ab = AdaBoostClassifier()

#parameters grid
param_grid_ab = {'n_estimators':list(range(1,30))}

#grid search
grid_ab = GridSearchCV(estimator_ab,param_grid_ab, cv=5, scoring='accuracy')
grid_ab.fit(X_train,y_train)

#best paramters for AdaBoost Model
grid_ab.best_params_

{'n_estimators': 1}

**AdaBoost Classifier**

**with important features & best Hyperparameters**

In [27]:
#input with important features
X_imp = X[important_features_list]


#train-test-split
X_train_ab,X_test_ab,y_train_ab,y_test_ab = train_test_split(X_imp,y,test_size=0.2,random_state=True)

#Modeling
from sklearn.ensemble import AdaBoostClassifier
ada = AdaBoostClassifier(n_estimators=1)
ada.fit(X_train_ab,y_train_ab)

#Evaluation on train data
ypred_train_ab = ada.predict(X_train_ab)

from sklearn.metrics import accuracy_score
print("Train accuracy:",accuracy_score(y_train_ab,ypred_train_ab))

#Cross validation
from sklearn.model_selection import cross_val_score
print("Cross Validation Score:",cross_val_score(ada,X_train_ab,y_train_ab,cv=5).mean())

#evaluation of Test data
ypred_test_ab = ada.predict(X_test_ab)
print("Test Accuracy:",accuracy_score(y_test_ab,ypred_test_ab))

Train accuracy: 0.9511274509803922
Cross Validation Score: 0.9511274509803922
Test Accuracy: 0.9494117647058824


**Gradient Boost**

**Applying Hyperparameter Tuning for Identifying best parameters for Gradient boost**



In [29]:
from sklearn.model_selection import GridSearchCV

#Model/estimator
from sklearn.ensemble import GradientBoostingClassifier
estimator_gb = GradientBoostingClassifier()

#parameters grid
param_grid = {'n_estimators':[1,2,4,6,8],
              'learning_rate':[0.1,0.2,0.3,0.5]}

#grid search
grid_gb = GridSearchCV(estimator_gb, param_grid, cv=5,scoring='accuracy')
grid_gb.fit(X_train,y_train)

#best parameters for GradientBoost Model
grid_gb.best_params_

{'learning_rate': 0.1, 'n_estimators': 1}

**Gradient Boost Classifier**

**with important features & best hyperparameters**



In [31]:
X_train_gb, X_test_gb, y_train_gb, y_test_gb = train_test_split(X_imp, y, test_size=0.2,random_state=True)

#modelling
from sklearn.ensemble import GradientBoostingClassifier
gb = GradientBoostingClassifier(n_estimators=1,learning_rate=0.1)
gb.fit(X_train_gb,y_train_gb)

#evaluation
ypred_train = gb.predict(X_train_gb)
print("Train accuracy:",accuracy_score(y_train_gb,ypred_train))

#Cross Validation
from sklearn.model_selection import cross_val_score
print("Cross Validation Score:",cross_val_score(gb,X_train_gb,y_train_gb,cv=5).mean())

#Evaluation of Test data
ypred_test = gb.predict(X_test_gb)
print("Test accuracy:",accuracy_score(y_test_gb,ypred_test))

Train accuracy: 0.9511274509803922
Cross Validation Score: 0.9511274509803922
Test accuracy: 0.9494117647058824


**XGBoost**

**Applying Hyperaparameter Tuning for Identifying best Parameters for xgboost**



In [33]:
from sklearn.model_selection import GridSearchCV

#model/estimator
from xgboost import XGBClassifier
estimator_xgb = XGBClassifier()

#parameters grid
param_grid = {'n_estimators':[10,20,30,40,50],
              'max_depth':[2,3,4,5,6],
              'gamma':[0,0.1,0.21,0.4,0.6]}

#grid search
grid_xgb = GridSearchCV(estimator_xgb,param_grid,cv=5,scoring='accuracy')
grid_xgb.fit(X_train,y_train)

#best parameters for XGB Model
grid_xgb.best_params_

{'gamma': 0, 'max_depth': 2, 'n_estimators': 10}

**XGB Model Important features**

In [35]:
X_imp = X[important_features_list]

X_train_xgb, X_test_xgb, y_train_xgb, y_test_xgb = train_test_split(X_imp, y, test_size=0.2,random_state=True)

#modelling
from xgboost import XGBClassifier
xgb = XGBClassifier(gamma=0,max_depth=2, n_estimators=10)
xgb.fit(X_train_xgb,y_train_xgb)

#Evaluation
ypred_train = xgb.predict(X_train_xgb)
print("Train Accuracy:",accuracy_score(y_train_xgb,ypred_train))

#Cross Validation
print("Cross Validation Score:",cross_val_score(xgb,X_train_xgb, y_train_xgb,cv=5).mean())

#Evaluation of Test data
ypred_test = xgb.predict(X_test_xgb)
print("Test Accuracy:",accuracy_score(y_test_xgb,ypred_test))

Train Accuracy: 0.9511274509803922
Cross Validation Score: 0.9511274509803922
Test Accuracy: 0.9494117647058824


In [70]:
import joblib

# Assuming 'final_rf_model' is your trained Random Forest model
joblib.dump(final_rf_model, 'fraud_detection_model.pkl')



['fraud_detection_model.pkl']